# Hicks-Ray CSTR: the advanced-step correction

Advanced-step NMPC runs the expensive solve *between* samples, at a
predicted state, and corrects it the instant the measurement arrives: a
backsolve on the solve's kept factorization, not a re-solve. This
notebook solves the Hicks-Ray CSTR at a prediction, writes a perturbed
measurement into the feedback hooks, and reads the corrected solution
against a full re-solve, accuracy first and then the timing that
motivates the idea.

In [1]:
import time

import pyomo.environ as pyo

import drto
from models.hicks import hicks

m = hicks(N=20)
pyo.TransformationFactory("dae.collocation").apply_to(
    m, wrt=m.t, nfe=20, ncp=3, scheme="LAGRANGE-RADAU")
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)
drto.info(m)

horizon,"t (ContinuousSet, 61 points)"
states,"zc (free), zt (free)"
dynamics,dzc[t] == (1 - zc[t])/(u2sf*v2[t]) - k0*zc[t]*exp(- ea/zt[t]) for t in t
dynamics,dzt[t] == (ztf - zt[t])/(u2sf*v2[t]) + k0*zc[t]*exp(- ea/zt[t]) - a0*u1sf*v1[t]*(zt[t] - ztcw) for t in t
controls,"v1 (piecewise_constant, free), v2 (piecewise_constant, free)"
tracking stage cost,cost[t] == 10*(zc[t] - zc_ss)**2 + 2*(zt[t] - zt_ss)**2 + (v1[t] - v1_ss)**2 + 0.5*(v2[t] - v2_ss)**2 for t in sorted(t)[:-1]
terminal cost,term == 10*(zc[20] - zc_ss)**2 + 2*(zt[20] - zt_ss)**2
initial conditions,zc[0] == zc_hat
initial conditions,zt[0] == zt_hat
steady-state targets,"zc_ss (of zc), zt_ss (of zt)"
steady-state control targets,"v1_ss (of v1), v2_ss (of v2)"


## Solve at the prediction

The assembly declared the feedback hooks as pounce sensitivity
parameters (the transformation log above records it), so an ordinary
pounce solve keeps the converged factorization.

In [2]:
tic = time.perf_counter()
pyo.SolverFactory("pounce").solve(m)
t_solve = time.perf_counter() - tic
i0 = sorted(m.v1)[0]
print(f"solve at the prediction: {t_solve:.3f} s")
print(f"first moves at the prediction: v1 = {pyo.value(m.v1[i0]):.5f}, "
      f"v2 = {pyo.value(m.v2[i0]):.5f}")

solve at the prediction: 0.182 s
first moves at the prediction: v1 = 0.42322, v2 = 0.58724


## The measurement arrives

Write the measured state into the hooks and ask for the corrected
solution. The model itself is untouched: the solution at the prediction
stays in place as the next solve's warm start.

In [3]:
m.zc_hat.set_value(0.635)  # predicted 0.625
m.zt_hat.set_value(0.515)  # predicted 0.525

tic = time.perf_counter()
est = drto.advanced_step_controller(m)
t_corr = time.perf_counter() - tic
print(f"correction: {1e3 * t_corr:.1f} ms "
      f"({t_solve / t_corr:.0f}x faster than the solve)")
print(f"corrected first moves:         v1 = {est[m.v1[i0]]:.5f}, "
      f"v2 = {est[m.v2[i0]]:.5f}")

correction: 3.4 ms (54x faster than the solve)
corrected first moves:         v1 = 0.32893, v2 = 0.67579


## Against the full re-solve

The correction is first order, so on this nonlinear model it is not
exact; the question is how close it lands.

In [4]:
mt = hicks(N=20)
pyo.TransformationFactory("dae.collocation").apply_to(
    mt, wrt=mt.t, nfe=20, ncp=3, scheme="LAGRANGE-RADAU")
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(mt)
mt.zc_hat.set_value(0.635)
mt.zt_hat.set_value(0.515)
pyo.SolverFactory("pounce").solve(mt)

print(f"re-solved first moves:         v1 = {pyo.value(mt.v1[i0]):.5f}, "
      f"v2 = {pyo.value(mt.v2[i0]):.5f}")
worst = 0.0
for comp in (m.zc, m.zt, m.v1, m.v2):
    for vd in comp.values():
        twin = mt.find_component(vd.name)
        worst = max(worst, abs(est[vd] - pyo.value(twin)))
print(f"largest deviation from the re-solve, all states and moves: "
      f"{worst:.2e}")

re-solved first moves:         v1 = 0.32420, v2 = 0.66643
largest deviation from the re-solve, all states and moves: 9.37e-03


## Sensitivities

`gradient=True` returns the controls' sensitivities to the hooks, the
raw material for analysis or a custom update law.

In [5]:
g = drto.advanced_step_controller(m, gradient=True)
for hook in (m.zc_hat, m.zt_hat):
    G1, G2 = g[m.v1][hook], g[m.v2][hook]
    print(f"d v1[{i0:.3f}] / d {hook.name} = {G1[m.v1[i0], hook]:+.4f}    "
          f"d v2[{i0:.3f}] / d {hook.name} = {G2[m.v2[i0], hook]:+.4f}")

d v1[0.000] / d zc_hat = +0.7045    d v2[0.000] / d zc_hat = +1.6211
d v1[0.000] / d zt_hat = +10.1333    d v2[0.000] / d zt_hat = -7.2341
